# UMU-Bench track: text-only unlearning of UMU's released vanilla

**What this runs.** Start model = UMU-Bench's fine-tuned LLaVA-1.5-7B (`chengyewang/llava_smu_ft`). We unlearn the 50
forget people with **name-based questions only (no images)**, then ask the same facts by name, by portrait, and by
name + portrait. The leak is **G = portrait accuracy − name accuracy** on facts the vanilla knew both ways
(*doubly-correct*). Then we look inside the model at one checkpoint (Milestone J tool checks, logit lens).

**Where things go.** Everything is written to Google Drive `MyDrive/vlm_unlearn/`: unlearning runs in
`runs/llava/umu_unlearn/<run_id>/` (LoRA per checkpoint in `ckpt/step<N>/`), mechanism results in
`results/llava/mech_umu/<run_id>/step<N>/`. Every cell is **resumable**: after a disconnect, run section 1 again and
then the cell you were on.

**Order.** 1 setup (every session) → 2 vanilla check (done once) → 3 unlearning runs → 4 mechanism on the chosen
checkpoint. Decisions and results so far: `DEVIATIONS.md` DEV-011 and DEV-012.
The old MLLMU-Bench main track (own vanilla fine-tuning, pilot, grid) is in `colab/archive_mllmu_track.ipynb`.

## 1. Setup (run every session)

In [ ]:
# --- set once ---------------------------------------------------------------
REPO_URL = "https://github.com/xuzhongwm/VLMUnlearn.git"
BRANCH = "main"
# ---------------------------------------------------------------------------
from google.colab import drive
drive.mount("/content/drive")
import os
os.environ["VLMU_ROOT"] = "/content/drive/MyDrive/vlm_unlearn"   # checkpoints, logs, results
os.environ["VLMU_LOCAL_CACHE"] = "/content/vlmu_cache"            # fast local disk (UMU images, cached data)
os.environ["HF_HOME"] = "/content/hf_home"                        # base model weights (local disk)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.makedirs(os.environ["VLMU_ROOT"], exist_ok=True)

In [ ]:
import os
if not os.path.exists("/content/VLMUnlearn"):
    !git clone -b {BRANCH} {REPO_URL} /content/VLMUnlearn
else:
    !git -C /content/VLMUnlearn pull --ff-only
%cd /content/VLMUnlearn
!git log --oneline -1

In [ ]:
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -c "import torch, transformers, peft; print(torch.__version__, transformers.__version__, peft.__version__, torch.cuda.is_available())"

## 2. Vanilla check: UMU's checkpoint scored the way UMU-Bench scores it (done; ~5 min on L4)
Generation + exact match on the forget set (100 facts × name / portrait) and the first 20 retain people.
Result 2026-11-05: forget name 0.75, portrait 0.81, both 0.65; AccF 0.823 = paper Table 1 (0.8233).
Only needed again if the readout code changes.

In [ ]:
!python scripts/eval_external.py --config configs/llava/vanilla.yaml --umu-protocol

## 3. Unlearning runs (name-based questions only, no images)
Each run gets its own folder (run id = method, lr, seed, optional tag) and evaluates at step 0 and every half or full epoch.
**How to read the final table** (all on doubly-correct facts, 1.00 at step 0):
- `forget dc txt` — asked by name: should fall (this is what we unlearn)
- `img` — asked by portrait; `t+i` — name + portrait; **`G` = img − txt** is the leak
- `retain dc txt / img` — other people: should stay near 1.00 (collateral damage)
- `degen` — share of the most common name-only answer: near 1.00 means the model says one thing to everything (collapse)

### 3a. Run 1: NPO only, lr 5e-5 (done)
Collapsed after epoch 1 (every name-only answer became "PAPAPA…"); only step 75 is usable. Run id `umu_npo_txt_lr5e-05_s0`.

In [ ]:
!python scripts/unlearn_umu.py --config configs/llava/umu_npo.yaml

### 3b. Run 2: NPO + retain (50 other people), lr 5e-5, 2 epochs (done; ~20 min on L4)
Step 150: forget name 0.63, portrait 0.97, name + portrait 0.97, **G 0.34**; retain name 0.69; no collapse.
Run id `umu_npo_rt_txt_lr5e-05_s0`. (The same config at lr 2e-5 barely forgot: `umu_npo_rt_txt_lr2e-05_s0`, stopped.)

In [ ]:
!python scripts/unlearn_umu.py --config configs/llava/umu_npo_rt.yaml --set train.lr=5.0e-5

### 3c. Run 3: stronger, broader retain (weight 3, 200 people), lr 5e-5 (done)
Step 150: forget name 0.92, portrait 0.97, G 0.05; retain name 0.96. Other people were protected, but the forgetting almost
vanished too. Run id `umu_npo_rt_txt_lr5e-05_s0_rw3n200`.


In [ ]:
!python scripts/unlearn_umu.py --config configs/llava/umu_npo_rt.yaml --set train.lr=5.0e-5 unlearn.retain_weight=3 umu.n_retain_unlearn=200 run_tag=rw3n200

### 3d. Run 4: same retain (weight 3, 200 people), lr 1e-4 (next; ~22 min on L4)
Question: is the damage to *other* people asked by name part of how this unlearning works, or just too little protection?
The readout now also has **seen** retain people (20 of the 200 the retain term trains on) next to the **unseen** 20.
- forget name falls, unseen retain falls, seen retain stays → the forgetting spreads to anyone asked by name; only people
  named in the retain set escape (it is the mechanism)
- forget name falls, both retain groups stay → person-specific forgetting is possible (run 3 was just too weak)
Run id `umu_npo_rt_txt_lr1e-04_s0_rw3n200`.

In [ ]:
!python scripts/unlearn_umu.py --config configs/llava/umu_npo_rt.yaml --set train.lr=1.0e-4 unlearn.retain_weight=3 umu.n_retain_unlearn=200 run_tag=rw3n200

## 4. Mechanism on one checkpoint (~5 min on L4)
Set the run and the step first (the step must be one of the run's evaluated checkpoints, e.g. 38 / 76 / 114 / 150).
Done so far: run 1 step 75, run 2 step 150.

In [ ]:
RUN_ID = "umu_npo_rt_txt_lr5e-05_s0"   # a folder under runs/llava/umu_unlearn/
STEP = 150
CONFIG = "configs/llava/umu_npo_rt.yaml"   # supplies the UMU settings only; RUN_ID picks the run
LENS = f"/content/drive/MyDrive/vlm_unlearn/results/llava/mech_umu/{RUN_ID}/step{STEP}"
print(RUN_ID, STEP)

### 4a. Milestone J: are the hook tools exact on this model + LoRA?
On an L4 (bf16) the log ends with "MILESTONE J FAILED" because two subtraction checks, (ii) and (iii-b)[all], miss their 1%
tolerance by bf16 rounding (top-1 agreement ≈ 0.98). That is expected. Everything else must PASS, and
`txt/J1_token_groups` must show `n_name > 0`. On an A100, drop `--set mech.dtype=bfloat16` to run the full check in float32.

In [ ]:
!python scripts/check_hooks_umu.py --config {CONFIG} --run-id {RUN_ID} --step {STEP} --set mech.dtype=bfloat16

### 4b. Logit lens + figure
Per layer: how much the unlearning moved log p(correct answer), for name / portrait / name + portrait. Writes `lens.json` and
the figure `lens_figure_first.png` (also as PDF) under `LENS`. The second cell only redraws the figure.

In [ ]:
!python scripts/mech_umu_lens.py --config {CONFIG} --run-id {RUN_ID} --step {STEP}
!python scripts/plot_umu_lens.py --lens {LENS}/lens.json
from IPython.display import Image, display
display(Image(f"{LENS}/lens_figure_first.png"))

In [ ]:
!python scripts/plot_umu_lens.py --lens {LENS}/lens.json
from IPython.display import Image, display
display(Image(f"{LENS}/lens_figure_first.png"))

### 4c. Layer masking: H1 (late) or H1′ (early)? (~5 min on L4, no training)
Switches the unlearning LoRA off on some layers and checks whether the name-only answers come back.
Left plot: switch off one 4-layer window at a time. Right plot: switch off a whole band (0–9, 10–19, 20–31), and keep a band alone.
**Read:** if the late band (20–31) both restores the answers when off and produces the suppression on its own → **H1**;
if the early band (0–9) does → **H1′**. The log also prints generation accuracy on the facts the name route lost.

In [ ]:
!python scripts/mech_umu_layer_mask.py --config {CONFIG} --run-id {RUN_ID} --step {STEP}
from IPython.display import Image, display
display(Image(f"{LENS}/layer_mask.png"))